# Agenda

1. Introducing `uv`
2. Running programs
3. venvs (virtual environments) and dependencies
4. `pyproject.toml` and `uv.lock`
5. Development dependencies'
6. Tools
7. Wheels
8. Publishing to PyPI with `uv`

# Background

Python has long been a language that claims to be "batteries included." By that, we mean that when you download and install Python, you have everything you need to get started and work.

That used to be true, and the standard library that comes with Python is huge and useful.

But a lot more packages are now on PyPI. You can download and install 846,000+ projects to your computer and use them right away.

Let's say that you want to download and install something from PyPI. The traditional way to do that is with `pip`. 

    pip install abcde

That would install the package on your computer and then you can say, in your program

    import abcde

and it's available!

BUT... what if you're working on two different projects? What if one project needs `abcde` version 1.0 and the other needs `abcde` version 2.0?

That's an issue because `pip` installs things into one, global directory, the `site-packages` directory. That is the final element in the `sys.path` list of strings, telling Python where to look for packages.

If you install version 1.0, then one of your projects won't work. If you install version 2.0, then the other project won't work.

The solution over the years has been to use "virtual environments," so that you can install them. (More on this later.)

Let's ignore conflicts... what about those global packages? You can and will install them with `pip`, and things seemed good.

Python packaging was always a bit complicated, between `pip`, and `site-packages`, and these conflicts... and it would often take a long time to double-check what packages you had installed, what packages worked with others, what packages needed each other, etc.

Every single Python conference I attended would have at least one talk about packaging in Python, and explaining all of the intricacies, and what you can/should do, etc.

About 2 years ago, I started to hear about `uv`. I heard that it was a "much faster version of `pip`." I started to play with it. It was definitely faster than `pip`! So what? `pip` was good, and `uv` was good.

Then... a student of mine said that if I'm only using it as a faster `pip`, I'm missing the point.

Basically, `uv` is a Python pacakaging super-app. It does everything. It replaces every tool you've ever heard of having to do with Python packaging, both using, creating, and distributing. It also replaces Python version management.

So it's instead of `pip`, `pipx`, `pyenv`, and `poetry`, and `twine`... You just need `uv`, and that's it.

`uv` is written in Rust, which runs way faster than Python. Plus, they've spent a lot of time trying to make it ultra-fast, so that you (and everyone) will use it all of the time. 

# Who writes `uv`?

You might be familiar with `ruff`, the super-fast Python linter and formatter and checker. Ruff is from Astral, a company that is dedicated to Python tools that are useful and fast. About 1-2 years ago, they also started to work on `uv`. 

About 2-3 months ago, Astral was bought by OpenAI.

Remember:

- All of Astral's code is open source
- Astral has committed to keeping everything open source
- Astral was a VC-backed company

Some people have said that they're going to stop using `uv` (or `ruff`) because of the OpenAI purchase. If and when they revoke the open-source license, or have `uv+` for paid customers, or whatever, I'll re-evaluate. 

# Very very basics with `uv`

The gateway drug version of `uv` is just to replace `pip`. It runs far, far faster than `pip` does. So why should you use it? Because it's faster.

It was easily 10x faster, and probably 100x faster.

# If not, then where?

The Python world has been moving, slowly but surely, over the last decade or so toward "projects." (Packages are collections of Python modules, with some metadata. Projects are directories/folders in which your Python code and a lot of related stuff, including a project definition file, are located.)

`uv` assumes that you will want to work inside of a project. You'll want to have lots of projects, each of which has its own configuration. 

To create a new project using `uv`, you use `uv init`:

- `uv init` inside of a directory means, "Turn this directory into a Python project."
- `uv init NAME` creates a new subdirectory (wherever you are) called `NAME`, and makes that a project.

# Exercise: Create a new project

1. Install `uv` -- follow directions here: https://docs.astral.sh/uv/getting-started/installation/
2. Go into one of your directories (maybe Desktop) and use `uv init NAME` to create a new project
3. Look around in the project -- see what files/folders you have

# `uv` installs Python, too!

If you download + install `uv`, and then you create a new project, `uv` will make sure that Python is installed and available.

# Now let's run something!

How do I run a program (let's say `main.py`) inside of my project?

The simple answer is: `python main.py`.

Don't do this. This is the *wrong* way to do things with `uv`. You now want to run programs with `uv run PROGRAM`. 

Why?

- `uv` will make sure you're using the local version of Python for your project.
- You can pass any arguments you want to `uv run`.

# Pinning the Python version

You can say

    uv python pin VERSION

to change `.python-version` in the home directory of your Python project. This determines what version is used when actually running. It must be compatiable with `python-version` in `pyproject.toml`, and `uv` won't let you set one that is incompatible.     

# Exercise: Write a short program and run it

1. In your project, create a small Python program.
2. Use `uv run` to run that program.
3. Try pinning 3.13 -- does that work?

# Next up

1. venvs (virtual environments) and dependencies
2. `pyproject.toml` and `uv.lock`


# Let's go back to conflicts

You are working on two different projects, X and Y. And project X uses `abcde` version 1, and project Y uses `abcde` version 2. There can be no compromise on these, and they are incompatible.

A problem is that Python doesn't let you install more than one version of a package into `site-packages`. You can imagine a version of Python with an `import` that lets you choose which version you want to load from a multi-versioned `site-packages`. That doesn't exist.

The solution has long been: Virtual environments, aka "venvs".

A venv has *nothing* to do with virtual machines, or cloud computing. Instead, a venv is just a sneaky trick that tells Python where your `site-packages` directory is, and to ignore the global one that everyone is using.

If you're using a venv:

- When you say `import` in your program, you get the packages from a venv-specific version of `site-packages`
- When you install a package, it goes into that package-specific directory

How does this work? It works via environment variables:

- When you enter the project directory with a venv, you "activate" it. (You often have to do with manually.)
- As soon as you do that, then the `site-packages` directory from your computer is ignored.
- When you're done working on your project, you run the `deactivate` function, and you're back to the global `site-packages`.

For years, people used venvs, and this way, they kept different projects separate. You could then install, for project X, `abcde` version 1 (you would say `pip install abcde==1.0`) and in a separate venv, you would install `abcde` version 2.

We still need venvs; we can't ignore them.

But when you use `uv run`, it automatically runs things within your local venv. Using venvs with `uv` is great, because you never have to think about them. `uv` takes care of looking for the pakcages in the right place. No activate, no deactivate, just coding and running things.

# How do we install packages?

The whole point of this is that when we install packages from PyPI, they'll be put into our venv's site-packages directory. How do we install them if we're not supposed to use pip?

Answer: Use `uv add`. This is the `uv`-native, super fast way of installing packages.

# `uv.lock`

In `pyproject.toml`, we say what top-level dependencies our project has:

- What packages do we want, and what versions?

But it doesn't cover:

- What are those packages depending on?
- What specifically should be downloaded from PyPI for that dependency

`uv.lock` is where `uv` figures out what version of each package on which you're dependent should be downloaded and installed for every possible combination of hardware, OS, and platform. It figures out which of those packages will work with which others.

You should never, ever edit `uv.lock`. You should, however, include it in your Git repo. That way, every single developer who works on your project (and users who download it) will be guaranteed that the dependencies will not only fit your project's requirementes, but also be compatible with one another.

# Exercise: Add dependencies

1. Take your sample program from before, and add functionality that's in a PyPI package. Use `import` to import it.
2. Try to run it with `uv run`. You will get an error, saying that the package isn't installed.
3. Use `uv add` to install the package.
4. Before you run your program, check `pyproject.toml` and `uv.lock`, and you'll see that it has been installed.
5. Now run it with `uv run` again.

# Relationship between `pyproject.toml`, `uv.lock` and `.venv`

The idea is as follows:

- You specify in `pyproject.toml` what packages you're going to `import` in your project. You'll also specify a version (ideally, both minimum and maximum).
- `uv` from your `pyproject.toml` file, creates `uv.lock`. That is the explicit, clear description of what files can be downloaded for each platform in order for your project to be installed and run somewhere.
- From `uv.lock`, `uv` then decides what to install in `.venv`

You really shouldn't be messing with `uv.lock` or `.venv` -- just update your `pyproject.toml`, and `uv` will do the rest.

But. What if `pyproject.toml` and `uv.lock` somehow get ... out of sync?

You can (and this is very rare, but you might need to) run `uv sync`. That looks at `pyproject.toml`, and goes through all of the resolution and rewrites `uv.lock` as necessary. Then it updates `.venv` with the appropriate files.

You can also (if you want, but it's even rarer) say `uv sync --frozen`. That tells `uv` that your `uv.lock` file is fine, and doesn't need changing/updating. However, it will then go and update things, as needed, in `.venv`.


- Command is `uv sync`
- File is `uv.lock`

Separately: All `uv` commands start with `uv` and then at least one word. This is similar to how `git init` or `git checkout` or `git commit` works -- it's just `uv THIS` and `uv THAT`.